In [1]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, ParameterSampler

import joblib

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False


In [2]:

LABEL = "fraud"

TRAIN_PATH = "../../5DATA/dataset/TRAIN_STAGE1"
TEST_PATH  = "../../5DATA/dataset/TEST_STAGE1"

OUT_DIR = "artifacts/stage1_models"
OUT_DIR_METRICS = "artifacts/stage1_metrics"


In [3]:

from pathlib import Path

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR_METRICS).mkdir(parents=True, exist_ok=True)

In [4]:

def load_stage_df(path: str, label: str = LABEL):
    df = pd.read_parquet(path)
    if label not in df.columns:
        raise KeyError(f"Missing label column: {label}")
    X = df.drop(columns=[label])
    y = df[label].astype(np.int8).to_numpy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X, y


In [5]:
def topk_metrics(y_true, score, top_pct_list=(0.001, 0.002, 0.005, 0.01)):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)
    n = len(y)
    base_rate = float(y.mean()) if n else np.nan
    order = np.argsort(-s)
    y_sorted = y[order]

    rows = []
    for p in top_pct_list:
        k = max(int(np.ceil(n * p)), 1)
        top_y = y_sorted[:k]
        prec = float(top_y.mean())
        rec = float(top_y.sum() / max(y.sum(), 1))
        lift = float(prec / base_rate) if base_rate and base_rate > 0 else np.nan
        rows.append({"top_pct": p, "k": k, "precision": prec, "recall": rec, "lift": lift, "base_rate": base_rate})
    return pd.DataFrame(rows)


def evaluate_metrics(y_true, score):
    return {
        "auc": float(roc_auc_score(y_true, score)),
        "prauc": float(average_precision_score(y_true, score)),
        "base_rate": float(np.mean(y_true)),
    }


In [6]:

def fit_logit(X_tr, y_tr):
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
        ))
    ])
    model.fit(X_tr, y_tr)
    return model


def fit_hgb(X_tr, y_tr):
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=200,
        l2_regularization=0.0,
        random_state=42,
    )
    model.fit(X_tr, y_tr)
    return model


def fit_lgb_small(X_tr, y_tr):
    if not HAS_LGB:
        raise RuntimeError("lightgbm is not available in this environment.")
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        min_data_in_leaf=300,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    return model


def predict_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1 / (1 + np.exp(-s))
    return model.predict(X)


In [7]:
X_tr, y_tr = load_stage_df(TRAIN_PATH)
X_te, y_te = load_stage_df(TEST_PATH)

print("train:", X_tr.shape, "test:", X_te.shape)
print("base_rate train:", float(y_tr.mean()), "test:", float(y_te.mean()))


train: (609658, 12) test: (114209, 12)
base_rate train: 0.01082246111754459 test: 0.018352318994124806


In [8]:

from sklearn.metrics import classification_report

candidates = [
    ("logit", fit_logit),
    ("hgb", fit_hgb),
]
if HAS_LGB:
    candidates.append(("lgb_small", fit_lgb_small))

results = []
topk_all = {}
reports = {}

for name, fit_fn in tqdm(candidates, desc="Training Stage1 models"):
    model = fit_fn(X_tr, y_tr)
    score_te = predict_score(model, X_te)

    m = evaluate_metrics(y_te, score_te)
    m["model"] = name
    results.append(m)

    topk = topk_metrics(y_te, score_te)
    topk_all[name] = topk
    topk.to_csv(Path(OUT_DIR_METRICS) / f"{name}_topk.csv", index=False)

    joblib.dump(model, Path(OUT_DIR) / f"{name}.joblib")
    np.save(Path(OUT_DIR_METRICS) / f"{name}_test_scores.npy", score_te)

    thr = np.quantile(score_te, 0.99)
    y_pred = (score_te >= thr).astype(int)

    rep_txt = classification_report(y_te, y_pred, digits=4)
    reports[name] = rep_txt

    print("\n" + "=" * 80)
    print(f"[{name}] threshold=quantile(0.99) -> top 1% as positive")
    print(rep_txt)

results_df = pd.DataFrame(results).sort_values(["prauc", "auc"], ascending=False).reset_index(drop=True)
results_df


Training Stage1 models:   0%|          | 0/3 [00:00<?, ?it/s]


[logit] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.9889    0.9973    0.9931    112113
           1     0.7349    0.4008    0.5187      2096

    accuracy                         0.9863    114209
   macro avg     0.8619    0.6990    0.7559    114209
weighted avg     0.9842    0.9863    0.9844    114209


[hgb] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.9901    0.9985    0.9943    112113
           1     0.8556    0.4666    0.6039      2096

    accuracy                         0.9888    114209
   macro avg     0.9229    0.7326    0.7991    114209
weighted avg     0.9876    0.9888    0.9871    114209

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_

,auc,prauc,base_rate,model
0,0.975992,0.663072,0.018352,hgb
1,0.977522,0.653782,0.018352,lgb_small
2,0.962786,0.501534,0.018352,logit


In [9]:

for name, _ in candidates:
    print(name)
    display(topk_all[name])


logit


,top_pct,k,precision,recall,lift,base_rate
0,0.001,115,0.721739,0.039599,39.326863,0.018352
1,0.002,229,0.759825,0.083015,41.402143,0.018352
2,0.005,572,0.751748,0.205153,40.962031,0.018352
3,0.010,1143,0.734908,0.400763,40.044429,0.018352


hgb


,top_pct,k,precision,recall,lift,base_rate
0,0.001,115,0.991304,0.054389,54.015209,0.018352
1,0.002,229,0.995633,0.108779,54.251083,0.018352
2,0.005,572,0.994755,0.271469,54.203245,0.018352
3,0.010,1143,0.855643,0.466603,46.623157,0.018352


lgb_small


,top_pct,k,precision,recall,lift,base_rate
0,0.001,115,1.000000,0.054866,54.489027,0.018352
1,0.002,229,0.995633,0.108779,54.251083,0.018352
2,0.005,572,0.996503,0.271947,54.298506,0.018352
3,0.010,1143,0.832896,0.454198,45.383686,0.018352


In [10]:

results_df.to_csv(Path(OUT_DIR_METRICS) / "stage1_summary.csv", index=False)
print("saved:", str(Path(OUT_DIR_METRICS) / "stage1_summary.csv"))


saved: artifacts/stage1_metrics/stage1_summary.csv


In [11]:
OUT_DIR = "artifacts/stage1_lgb_tuned"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

---

In [12]:

def load_stage_df(path: str, label: str = LABEL):
    df = pd.read_parquet(path)
    if label not in df.columns:
        raise KeyError(f"Missing label column: {label}")
    X = df.drop(columns=[label]).replace([np.inf, -np.inf], np.nan).fillna(0)
    y = df[label].astype(np.int8).to_numpy()
    return X, y

X_tr_full, y_tr_full = load_stage_df(TRAIN_PATH)
X_te, y_te = load_stage_df(TEST_PATH)

print("train:", X_tr_full.shape, "test:", X_te.shape)
print("base_rate train:", float(y_tr_full.mean()), "test:", float(y_te.mean()))


train: (609658, 12) test: (114209, 12)
base_rate train: 0.01082246111754459 test: 0.018352318994124806


In [13]:

X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr_full, y_tr_full,
    test_size=0.2,
    random_state=42,
    stratify=y_tr_full
)

pos = float(np.sum(y_tr == 1))
neg = float(np.sum(y_tr == 0))
scale_pos_weight = (neg / max(pos, 1.0))

scale_pos_weight


91.40735126942023

In [14]:

def fit_eval_lgb(params, X_tr, y_tr, X_va, y_va):
    clf = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=5000,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        **params
    )

    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="average_precision",
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    score_va = clf.predict_proba(X_va)[:, 1]
    prauc = float(average_precision_score(y_va, score_va))
    auc = float(roc_auc_score(y_va, score_va))
    best_iter = int(getattr(clf, "best_iteration_", clf.n_estimators))
    return clf, prauc, auc, best_iter


In [15]:

param_space = {
    "learning_rate": [0.02, 0.03, 0.05, 0.07],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [-1, 6, 7, 8, 10],
    "min_data_in_leaf": [50, 100, 200, 300, 500],
    "feature_fraction": [0.6, 0.7, 0.8, 0.9, 1.0],
    "bagging_fraction": [0.6, 0.7, 0.8, 0.9, 1.0],
    "bagging_freq": [0, 1],
    "reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "reg_lambda": [0.0, 0.5, 1.0, 2.0, 5.0],
}

N_TRIALS = 30
sampler = list(ParameterSampler(param_space, n_iter=N_TRIALS, random_state=42))

best = {"prauc": -1.0, "auc": -1.0, "params": None, "best_iter": None, "model": None}
trial_rows = []

for i, params in tqdm(list(enumerate(sampler, start=1)), total=len(sampler), desc="Tuning LGBM (valid PR-AUC)"):
    model, prauc, auc, best_iter = fit_eval_lgb(params, X_tr, y_tr, X_va, y_va)
    trial_rows.append({
        "trial": i,
        "prauc_valid": prauc,
        "auc_valid": auc,
        "best_iter": best_iter,
        **params
    })

    if prauc > best["prauc"]:
        best.update({"prauc": prauc, "auc": auc, "params": params, "best_iter": best_iter, "model": model})

trials_df = pd.DataFrame(trial_rows).sort_values(["prauc_valid", "auc_valid"], ascending=False).reset_index(drop=True)
trials_df.head(10)


Tuning LGBM (valid PR-AUC):   0%|          | 0/30 [00:00<?, ?it/s]

[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Info] Number of positive: 5278, number of 

,trial,prauc_valid,auc_valid,best_iter,reg_lambda,reg_alpha,num_leaves,min_data_in_leaf,max_depth,learning_rate,feature_fraction,bagging_freq,bagging_fraction
0,6,0.931527,0.997181,223,5.0,1.0,63,300,10,0.07,1.0,0,0.7
1,23,0.930039,0.997171,347,5.0,0.1,63,300,7,0.07,0.6,0,1.0
2,16,0.929711,0.996810,526,1.0,0.1,31,200,-1,0.07,1.0,0,0.9
3,14,0.929589,0.996991,389,2.0,0.0,31,50,8,0.07,0.7,0,0.8
4,20,0.929280,0.996651,235,2.0,0.5,63,100,7,0.07,0.8,0,0.9
5,8,0.928995,0.997071,204,1.0,0.5,127,300,10,0.07,0.6,1,0.8
6,25,0.928767,0.997019,419,0.5,0.0,31,300,7,0.07,1.0,0,0.8
7,4,0.928669,0.997110,337,2.0,1.0,255,200,10,0.05,0.6,1,1.0
8,7,0.928663,0.997021,288,2.0,0.1,63,300,-1,0.07,0.9,0,0.7
9,11,0.928585,0.996886,308,2.0,1.0,127,300,8,0.07,0.6,0,0.7


In [16]:
best["prauc"], best["auc"], best["best_iter"], best["params"]

(0.9315269483278562,
 0.9971806413553503,
 223,
 {'reg_lambda': 5.0,
  'reg_alpha': 1.0,
  'num_leaves': 63,
  'min_data_in_leaf': 300,
  'max_depth': 10,
  'learning_rate': 0.07,
  'feature_fraction': 1.0,
  'bagging_freq': 0,
  'bagging_fraction': 0.7})

In [17]:

best_model = best["model"]
score_te = best_model.predict_proba(X_te)[:, 1]

print("TEST AUC:", float(roc_auc_score(y_te, score_te)))
print("TEST PR-AUC:", float(average_precision_score(y_te, score_te)))


[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
TEST AUC: 0.9777760474829001
TEST PR-AUC: 0.672372809856099


In [18]:

def report_at_top_pct(y_true, score, top_pct):
    thr = float(np.quantile(score, 1 - top_pct))
    y_pred = (score >= thr).astype(int)
    print(f"\nTop {top_pct*100:.3f}% (threshold={thr:.6f})")
    print(classification_report(y_true, y_pred, digits=4))

for p in [0.01, 0.005, 0.002, 0.001, 0.0005]:
    report_at_top_pct(y_te, score_te, p)



Top 1.000% (threshold=0.998752)
              precision    recall  f1-score   support

           0     0.9903    0.9987    0.9944    112113
           1     0.8696    0.4742    0.6138      2096

    accuracy                         0.9890    114209
   macro avg     0.9299    0.7365    0.8041    114209
weighted avg     0.9880    0.9890    0.9875    114209


Top 0.500% (threshold=0.999921)
              precision    recall  f1-score   support

           0     0.9866    1.0000    0.9932    112113
           1     0.9948    0.2715    0.4265      2096

    accuracy                         0.9866    114209
   macro avg     0.9907    0.6357    0.7099    114209
weighted avg     0.9867    0.9866    0.9828    114209


Top 0.200% (threshold=0.999975)
              precision    recall  f1-score   support

           0     0.9836    1.0000    0.9917    112113
           1     0.9913    0.1083    0.1953      2096

    accuracy                         0.9836    114209
   macro avg     0.9874    0.

In [19]:

joblib.dump(best_model, Path(OUT_DIR) / "lgb_stage1_tuned.joblib")
np.save(Path(OUT_DIR) / "lgb_stage1_tuned_test_scores.npy", score_te)
trials_df.to_csv(Path(OUT_DIR) / "lgb_tuning_trials.csv", index=False)

with open(Path(OUT_DIR) / "lgb_best_params.json", "w", encoding="utf-8") as f:
    import json
    json.dump(
        {"best_valid_prauc": best["prauc"], "best_valid_auc": best["auc"], "best_iter": best["best_iter"], "params": best["params"]},
        f, ensure_ascii=False, indent=2
    )

print("saved:", OUT_DIR)


saved: artifacts/stage1_lgb_tuned


### Stage1의 역할
> “확실히 아닌 애들은 통과시키고,
> 위험해 보이는 애들만 소수로 남긴다”

-> recall이 1순위 아님 (recall은 stage2가 보완)\
=> Precision이 더 중요하게 작용


In [20]:
MODEL_PATH = Path(OUT_DIR) / "lgb_stage1_tuned.joblib"

model = joblib.load(MODEL_PATH)
print("model loaded:", MODEL_PATH)


model loaded: artifacts/stage1_lgb_tuned/lgb_stage1_tuned.joblib


In [21]:
score_te = model.predict_proba(X_te)[:, 1]

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7


In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

def sweep_threshold_by_top_pct(y_true, score, pct_list=None):
  
    if pct_list is None:
        pct_list = np.linspace(0.1, 10, 100)

    y = np.asarray(y_true)
    s = np.asarray(score)

    df = pd.DataFrame({"y": y, "s": s})
    df = df.sort_values("s", ascending=False).reset_index(drop=True)

    n = len(df)
    results = []

    for pct in pct_list:
        k = int(n * pct / 100)
        if k <= 0:
            continue

        thr = df.loc[k-1, "s"]
        y_pred = (s >= thr).astype(int)

        prec = precision_score(y, y_pred, zero_division=0)
        rec  = recall_score(y, y_pred, zero_division=0)
        f1   = f1_score(y, y_pred, zero_division=0)

        results.append({
            "top_pct": pct,
            "threshold": thr,
            "precision": prec,
            "recall": rec,
            "f1": f1,
        })

    return pd.DataFrame(results)

def pick_best_under_recall(sweep_df, min_recall=0.5):
    """
    recall >= min_recall 조건 하에서 precision 최대인 row 선택
    """
    df = sweep_df[sweep_df["recall"] >= min_recall].copy()

    if df.empty:
        return None, sweep_df

    best_row = df.sort_values("precision", ascending=False).iloc[0]
    return best_row, df

In [23]:
sweep = sweep_threshold_by_top_pct(y_te, score_te)
best_row, _ = pick_best_under_recall(sweep, min_recall=0.50)

best_row


top_pct      1.200000
threshold    0.998344
precision    0.775182
recall       0.506679
f1           0.612810
Name: 11, dtype: float64

In [24]:
row = best_row  

thr = float(row["threshold"])

y_pred = (np.asarray(score_te) >= thr).astype(int)

print(f"Selected threshold = {thr:.6f}")
print(f"top_pct = {float(row['top_pct']):.6f} | "
      f"precision = {float(row['precision']):.4f} | "
      f"recall = {float(row['recall']):.4f}\n")

from sklearn.metrics import classification_report
print(classification_report(y_te, y_pred, digits=4))

Selected threshold = 0.998344
top_pct = 1.200000 | precision = 0.7752 | recall = 0.5067

              precision    recall  f1-score   support

           0     0.9908    0.9973    0.9940    112113
           1     0.7752    0.5067    0.6128      2096

    accuracy                         0.9882    114209
   macro avg     0.8830    0.7520    0.8034    114209
weighted avg     0.9869    0.9882    0.9870    114209



---

### id 추가해서 다시 돌린 후 stage1에서 1로 후보 행 parquet 생성

In [25]:
X_tr.columns

Index(['id', 'log_abs_amount', 'amount_deviation', 'client_fraud_last1',
       'card_fraud_last3', 'tx_hour', 'hour_cos', 'is_highrisk_weekday',
       'seconds_since_prev_tx', 'card_velocity_spike_ratio',
       'client_mcc_is_new', 'client_merchant_is_new'],
      dtype='object')

In [26]:
# id 추가해서 다시 돌린 후 stage1에서 1로 후보 행 parquet 생성

LABEL = "fraud"
ID_COL = "id"


def split_X_y_id(df: pd.DataFrame, label_col=LABEL, id_col=ID_COL):
    df = df.copy()
    if id_col not in df.columns:
        raise KeyError(f"missing {id_col}")

    y = df[label_col].astype("int8").to_numpy()
    ids = df[id_col].to_numpy()

    # label만 drop, id는 X에 남김
    X = df.drop(columns=[label_col])

    return X, y, ids

In [27]:
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
X_tr, y_tr, id_tr = split_X_y_id(train)
X_te, y_te, id_te = split_X_y_id(test)


In [28]:
MODEL_DIR = Path("artifacts/stage1_lgb_tuned")

best_model = joblib.load(MODEL_DIR / "lgb_stage1_tuned.joblib")

In [29]:
import numpy as np
import pandas as pd

def get_model_feature_names(m):
    # sklearn API 래퍼 (lightgbm>=3.3)에서는 feature_name_이 있음
    if hasattr(m, "feature_name_") and m.feature_name_ is not None:
        return list(m.feature_name_)
    # 내부 Booster에서 가져오기
    if hasattr(m, "booster_"):
        return m.booster_.feature_name()
    # 마지막 수단
    raise AttributeError("Cannot find feature names from model (feature_name_/booster_ not found).")

model_feats = get_model_feature_names(best_model)
cur_feats = list(X_te.columns)

missing_in_data = [c for c in model_feats if c not in cur_feats]  # 모델은 기대하는데 X에 없음
extra_in_data   = [c for c in cur_feats if c not in model_feats]  # X에는 있는데 모델 학습엔 없었음

print("model n_features:", len(model_feats))
print("X_te  n_features:", len(cur_feats))
print("\n[Missing in X_te] (model expects, but data lacks):")
print(missing_in_data)

print("\n[Extra in X_te] (data has, but model didn't train on):")
print(extra_in_data)

model n_features: 12
X_te  n_features: 12

[Missing in X_te] (model expects, but data lacks):
[]

[Extra in X_te] (data has, but model didn't train on):
[]


In [30]:
score_te = best_model.predict(X_te)


[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7


In [31]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

def find_best_threshold_under_recall(y_true, scores, min_recall=0.5):
    thresholds = np.linspace(0, 1, 500)
    best_thr = None
    best_precision = -1

    for thr in thresholds:
        y_pred = (scores >= thr).astype(int)
        recall = recall_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)

        if recall >= min_recall and precision > best_precision:
            best_precision = precision
            best_thr = thr

    return best_thr


In [32]:
thr = find_best_threshold_under_recall(y_te, score_te, min_recall=0.50)
print("Selected threshold:", thr)

Selected threshold: 0.002004008016032064


In [33]:
pass_mask = score_te >= thr

pass_ids = pd.Series(id_te[pass_mask], name="id")

print("n_total:", len(id_te))
print("n_pass:", pass_mask.sum())
print("pass_rate:", pass_mask.mean())

pass_ids.head()

n_total: 114209
n_pass: 14051
pass_rate: 0.1230288331042212


0    17207495
1    17338826
2    17482807
3    17692865
4    17779020
Name: id, dtype: int64

In [34]:
out_df = test.loc[pass_mask, ["id"]].copy()
out_df.to_parquet("artifacts/stage1_pass_ids_test.parquet", index=False)